
# NB3 — Ablation Part 2 (α=0.8 | no-R-matrix) + Baselines (DenseNet121 | ViT-B/16)

GPU time: ~75-90 min. Run AFTER NB2. Trains 4 models. Saves logits + result JSONs to Drive.

**Before running:** Connect T4 GPU (Runtime → Change runtime type → T4 GPU)

**Drive folder:** `MyDrive/CNN_GNN_Results/` must exist (created automatically on first run).

In [ ]:
import os, hashlib, random, warnings, json, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             f1_score, accuracy_score)
from scipy.stats import chi2
import warnings; warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

CLASS_NAMES  = ['Cardiomegaly','Covid-19','Normal',
                'Pneumonia','Pneumothorax','Tuberculosis']
NUM_CLASSES  = 6
DATASET_PATH = '/content/drive/MyDrive/Dataset'
RESULTS_DIR  = '/content/drive/MyDrive/CNN_GNN_Results'
CKPT_PATH    = f'{RESULTS_DIR}/cnn_gnn_final.pth'
BATCH_SIZE   = 32
IMAGE_SIZE   = 224
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')


Mounted at /content/drive
Device: cuda
Setup complete.


In [ ]:
# ── New baseline models (required by reviewers) ───────────────


# ViT-B/16 — Vision Transformer baseline
class ViTBaseline(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.vit = models.vit_b_16(pretrained=True)
        # Freeze all; unfreeze last 2 transformer blocks + head
        for p in self.vit.parameters(): p.requires_grad = False
        for p in self.vit.encoder.layers[-2:].parameters(): p.requires_grad = True
        hidden = self.vit.heads.head.in_features
        self.vit.heads = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(hidden, num_classes))
    def forward(self, x): return self.vit(x)

print('Baseline model classes ready.')


Baseline model classes ready.


In [ ]:
# ── Dataset (same transforms as Untitled7) ───────────────────
class ChestXrayDataset(Dataset):
    def __init__(self, root_dir, transform, split):
        self.transform   = transform
        self.image_paths = []
        self.labels      = []
        self.patient_ids = []   # for leakage check
        split_path = os.path.join(root_dir, split)
        print(f'Loading [{split}]...')
        for idx, cls in enumerate(CLASS_NAMES):
            d = os.path.join(split_path, cls)
            if not os.path.exists(d): continue
            files = [f for f in os.listdir(d)
                     if f.lower().endswith(('.png','.jpg','.jpeg'))]
            for f in files:
                self.image_paths.append(os.path.join(d, f))
                self.labels.append(idx)
                self.patient_ids.append(hashlib.md5(f.encode()).hexdigest()[:8])
            print(f'  {cls:<16}: {len(files)}')
        print(f'  Total: {len(self.image_paths)}')

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, i):
        try: return self.transform(
            Image.open(self.image_paths[i]).convert('RGB')), self.labels[i]
        except: return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE), 0


train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

train_ds = ChestXrayDataset(DATASET_PATH, train_tf, 'train')
val_ds   = ChestXrayDataset(DATASET_PATH, val_tf,   'valid')
test_ds  = ChestXrayDataset(DATASET_PATH, val_tf,   'test')
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Patient leakage check
tr_p = set(train_ds.patient_ids)
va_p = set(val_ds.patient_ids)
te_p = set(test_ds.patient_ids)
print(f'Leakage check — Train∩Val:{len(tr_p&va_p)}  '
      f'Train∩Test:{len(tr_p&te_p)}  Val∩Test:{len(va_p&te_p)}')
print('(0 overlap = no leakage; pseudo-IDs from filename hash)')
print('Dataset ready.')


Loading [train]...
  Cardiomegaly    : 1800
  Covid-19        : 1800
  Normal          : 1800
  Pneumonia       : 1800
  Pneumothorax    : 1800
  Tuberculosis    : 1800
  Total: 10800
Loading [valid]...
  Cardiomegaly    : 225
  Covid-19        : 225
  Normal          : 225
  Pneumonia       : 225
  Pneumothorax    : 225
  Tuberculosis    : 225
  Total: 1350
Loading [test]...
  Cardiomegaly    : 225
  Covid-19        : 225
  Normal          : 225
  Pneumonia       : 225
  Pneumothorax    : 225
  Tuberculosis    : 225
  Total: 1350
Leakage check — Train∩Val:72  Train∩Test:149  Val∩Test:49
(0 overlap = no leakage; pseudo-IDs from filename hash)
Dataset ready.


In [ ]:
# ── Loss, metrics, training helpers ─────────────────────────
class MultiClassFocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        return ((1 - torch.exp(-ce)) ** self.gamma * ce).mean()


def get_metrics(logits_t, tgts_t):
    probs = F.softmax(logits_t, dim=1)
    preds = probs.argmax(1).numpy(); t = tgts_t.numpy()
    acc  = accuracy_score(t, preds)
    mf1  = f1_score(t, preds, average='macro', zero_division=0)
    pf1  = f1_score(t, preds, average=None,
                    labels=list(range(NUM_CLASSES)), zero_division=0)
    cm   = confusion_matrix(t, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for c in range(NUM_CLASSES):
        TP=cm[c,c]; FN=cm[c,:].sum()-TP; FP=cm[:,c].sum()-TP; TN=cm.sum()-TP-FN-FP
        sens.append(TP/(TP+FN+1e-8)); spec.append(TN/(TN+FP+1e-8))
    oh = F.one_hot(tgts_t, NUM_CLASSES).numpy(); auc = []
    for c in range(NUM_CLASSES):
        try: auc.append(roc_auc_score(oh[:,c], probs[:,c].numpy()))
        except: auc.append(float('nan'))
    return {'acc':acc,'mf1':mf1,'pf1':pf1,
            'sens':np.array(sens),'spec':np.array(spec),
            'auc':auc,'cm':cm,'probs':probs}


@torch.no_grad()
def evaluate(model, loader):
    model.eval(); all_logits, all_tgts = [], []
    for imgs, lbls in loader:
        all_logits.append(model(imgs.to(device)).cpu())
        all_tgts.append(lbls)
    logits = torch.cat(all_logits); tgts = torch.cat(all_tgts)
    return logits, tgts, get_metrics(logits, tgts)


def print_test_results(m, label='TEST'):
    print(f'\n{"="*55}\n{label}\n{"="*55}')
    print(f'Accuracy : {m["acc"]*100:.2f}%')
    print(f'Macro F1 : {m["mf1"]:.4f}')
    print(f'Mean AUC : {np.nanmean(m["auc"]):.4f}')
    print(f'Mean Sens: {m["sens"].mean():.4f}')
    print(f'Mean Spec: {m["spec"].mean():.4f}')
    print(f'\n{"Disease":<16} {"F1":>7} {"AUC":>7} {"Sens":>7} {"Spec":>7}')
    print('-'*46)
    for i, name in enumerate(CLASS_NAMES):
        print(f'{name:<16} {m["pf1"][i]:>7.4f} {m["auc"][i]:>7.4f} '
              f'{m["sens"][i]:>7.4f} {m["spec"][i]:>7.4f}')


class EarlyStopping:
    def __init__(self, patience=10):
        self.patience=patience; self.counter=0; self.best=float('inf')
    def __call__(self, v):
        if v < self.best-1e-4: self.best=v; self.counter=0; return False
        self.counter += 1; return self.counter >= self.patience


def quick_train(model, name, epochs=30, patience=10, lr=0.0001):
    """
    Train any model variant and save best checkpoint + result JSON.
    Matches Untitled7 training setup exactly.
    """
    save_path = f'/content/{name}.pth'
    criterion = MultiClassFocalLoss(gamma=2.0)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    stopper   = EarlyStopping(patience)
    best_f1   = 0.0

    for ep in range(1, epochs+1):
        # ── Train ──
        model.train()
        for imgs, lbls in tqdm(train_loader, desc=f'  Train E{ep:02d}', leave=False):
            loss = criterion(model(imgs.to(device)), lbls.to(device))
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # ── Validate ──
        val_logits, val_tgts, vm = evaluate(model, val_loader)
        val_loss = MultiClassFocalLoss()(val_logits, val_tgts).item()

        print(f'  E{ep:02d}  val_f1={vm["mf1"]:.4f}  '
              f'val_acc={vm["acc"]*100:.2f}%  val_loss={val_loss:.4f}')

        if vm['mf1'] > best_f1:
            best_f1 = vm['mf1']
            torch.save(model.state_dict(), save_path)
            print(f'       -> Best saved')

        if stopper(val_loss):
            print(f'  Early stop at epoch {ep}'); break

    # Load best weights
    model.load_state_dict(
        torch.load(save_path, map_location=device, weights_only=True))

    # Test evaluation
    logits, tgts, m = evaluate(model, test_loader)
    print_test_results(m, label=f'{name} TEST')

    # Save JSON result summary
    result = {
        'model': name,
        'acc':   float(m['acc']),
        'mf1':   float(m['mf1']),
        'auc':   float(np.nanmean(m['auc'])),
        'sens':  float(m['sens'].mean()),
        'spec':  float(m['spec'].mean()),
        'trainable_params': sum(p.numel() for p in model.parameters()
                                if p.requires_grad)
    }
    with open(f'{RESULTS_DIR}/{name}_result.json', 'w') as f:
        json.dump(result, f, indent=2)

    # Copy checkpoint to Drive
    shutil.copy(save_path, f'{RESULTS_DIR}/{name}.pth')
    print(f'  Saved {name}.pth + {name}_result.json to Drive.')
    return model, logits, tgts, m


print('All helpers ready.')


All helpers ready.


In [ ]:
# ── Baseline: ViT-B/16 ─────────────────────────────────────────────────
print('\n' + '='*55)
print('Baseline: ViT-B/16')
print('='*55)
base_vit = ViTBaseline(NUM_CLASSES).to(device)
tr_base_vit = sum(p.numel() for p in base_vit.parameters() if p.requires_grad)
print(f'Trainable params: {tr_base_vit:,}')

base_vit, lg_base_vit, tg_base_vit, m_base_vit = quick_train(
    base_vit, 'base_vit', epochs=30, patience=10)

# Save logits for McNemar test in NB4
torch.save({'logits': lg_base_vit, 'tgts': tg_base_vit},
           f'{RESULTS_DIR}/base_vit_logits.pth')
del base_vit; torch.cuda.empty_cache()
print(f'Baseline: ViT-B/16 done.')



Baseline: ViT-B/16
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 161MB/s]


Trainable params: 14,180,358


  E01  val_f1=0.9266  val_acc=92.67%  val_loss=0.0747
       -> Best saved


  E02  val_f1=0.9212  val_acc=92.00%  val_loss=0.0884


...

  E12  val_f1=0.9631  val_acc=96.30%  val_loss=0.0510


  E13  val_f1=0.9655  val_acc=96.52%  val_loss=0.0602


  E14  val_f1=0.9532  val_acc=95.26%  val_loss=0.0855


  E15  val_f1=0.9705  val_acc=97.04%  val_loss=0.0579
       -> Best saved


  E16  val_f1=0.9585  val_acc=95.78%  val_loss=0.0748
  Early stop at epoch 16

base_vit TEST
Accuracy : 97.63%
Macro F1 : 0.9763
Mean AUC : 0.9986
Mean Sens: 0.9763
Mean Spec: 0.9953

Disease               F1     AUC    Sens    Spec
----------------------------------------------
Cardiomegaly      0.9933  1.0000  0.9867  1.0000
Covid-19          0.9669  0.9978  0.9733  0.9920
Normal            0.9518  0.9966  0.9644  0.9876
Pneumonia         0.9866  1.0000  0.9822  0.9982
Pneumothorax      0.9661  0.9971  0.9511  0.9964
Tuberculosis      0.9934  1.0000  1.0000  0.9973
  Saved base_vit.pth + base_vit_result.json to Drive.
Baseline: ViT-B/16 done.


In [ ]:
# ── Partial results table ─────────────────────────────────────
rows = []
for name, fname in [('A4: GNN α=0.8','abl_alpha08'),
                    ('A5: GNN no R-matrix','abl_no_rel'),
                    ('DenseNet121 (CheXNet)','base_dn121'),
                    ('ViT-B/16','base_vit')]:
    p = f'{RESULTS_DIR}/{fname}_result.json'
    if os.path.exists(p):
        with open(p) as f: d = json.load(f)
        rows.append({'Model':name,'Acc(%)':f'{d["acc"]*100:.2f}',
                     'Macro F1':f'{d["mf1"]:.4f}','Mean AUC':f'{d["auc"]:.4f}',
                     'Sens':f'{d["sens"]:.4f}','Spec':f'{d["spec"]:.4f}',
                     'Params(M)':f'{d["trainable_params"]/1e6:.2f}'})
df = pd.DataFrame(rows)
print('\nNB3 RESULTS:')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/ablation_part2_and_baselines.csv', index=False)
print('Saved ablation_part2_and_baselines.csv to Drive.')
